In [1]:
import math

import tensorflow as tf
import keras_cv
from tensorflow import keras

ERROR:absl:cannot import name 'runtime_version' from 'google.protobuf' (d:\Users\Saulete\Downloads\venv\Lib\site-packages\google\protobuf\__init__.py)
Traceback (most recent call last):
  File "d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow_datasets\__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow_datasets\rlds\__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow_datasets\rlds\envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow_datasets\core\__init__.py", line 21, in <module>
    from tensorflow_datasets.core import community
  File "d:\Users\Saulete\Downloads\venv

You do not have pycocotools installed, so KerasCV pycoco metrics are not available. Please run `pip install pycocotools`.
You do not have pyococotools installed, so the `PyCOCOCallback` API is not available.
You do not have Waymo Open Dataset installed, so KerasCV Waymo metrics are not available.


In [2]:
# load the pipeline, get text_encoder and decoder

model = keras_cv.models.StableDiffusion(img_width=256, img_height=256)

tokenizer = model.tokenizer
text_encoder_model = model.text_encoder
decoder_model = model.decoder

By using this model checkpoint, you acknowledge that its usage is subject to the terms of the CreativeML Open RAIL-M license at https://raw.githubusercontent.com/CompVis/stable-diffusion/main/LICENSE


In [3]:
MAX_PROMPT_LENGTH = 77

def get_pos_ids():
    return tf.convert_to_tensor([list(range(MAX_PROMPT_LENGTH))], dtype=tf.int32)

def representative_data_gen_text_encoder():
    for i in range(1):
        inputs = tokenizer.encode('This is a test')
        phrase = inputs + [49407] * (MAX_PROMPT_LENGTH - len(inputs))
        phrase = tf.convert_to_tensor([phrase], dtype=tf.int32)

        yield [phrase, get_pos_ids()]
        
def representative_data_gen_decoder():
    for i in range(1):
        noise = tf.random.normal((1, 32, 32, 4))
        yield [noise]

In [4]:
# convert the two models to tflite

static_tokens = tf.keras.Input(batch_shape=(1, 77), dtype=tf.int32, name="tokens")
static_positions = tf.keras.Input(batch_shape=(1, 77), dtype=tf.int32, name="positions")

static_text_encoder_model = tf.keras.Model(
    inputs=[static_tokens, static_positions], 
    outputs=text_encoder_model([static_tokens, static_positions])
)

converter1 = tf.lite.TFLiteConverter.from_keras_model(static_text_encoder_model)
converter1.optimizations = [tf.lite.Optimize.DEFAULT]
converter1.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter1.target_spec.supported_types = [tf.int8]
converter1.inference_input_type = tf.uint8
converter1.inference_output_type = tf.uint8
converter1.representative_dataset = representative_data_gen_text_encoder
tflite_text_encoder_qint8 = converter1.convert()

with open('/tmp/sd_text_encoder_qint8.tflite', 'wb') as f:
    f.write(tflite_text_encoder_qint8)
    
static_input = tf.keras.Input(batch_shape=(1, 32, 32, 4))
static_decoder_model = tf.keras.Model(
    inputs=static_input, 
    outputs=decoder_model(static_input)
)
    
converter2 = tf.lite.TFLiteConverter.from_keras_model(static_decoder_model)
converter2.optimizations = [tf.lite.Optimize.DEFAULT]
converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter2.target_spec.supported_types = [tf.int8]
converter2.inference_input_type = tf.uint8
converter2.inference_output_type = tf.uint8
converter2.representative_dataset = representative_data_gen_decoder
tflite_decoder_qint8 = converter2.convert()
    
with open('/tmp/sd_decoder_qint8.tflite', 'wb') as f:
    f.write(tflite_decoder_qint8)

INFO:tensorflow:Assets written to: E:\Microsoft VS Code\data\tmp\tmp2c5t0kn4\assets


INFO:tensorflow:Assets written to: E:\Microsoft VS Code\data\tmp\tmp2c5t0kn4\assets
d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow\lite\python\convert.py:789: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "


INFO:tensorflow:Assets written to: E:\Microsoft VS Code\data\tmp\tmp6lrxqxj3\assets


INFO:tensorflow:Assets written to: E:\Microsoft VS Code\data\tmp\tmp6lrxqxj3\assets
d:\Users\Saulete\Downloads\venv\Lib\site-packages\tensorflow\lite\python\convert.py:789: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "
